In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models



In [2]:
# ✅ GPU setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(256, scale=(0.9, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

In [4]:
data_dir = r"C:\Users\harsh\OneDrive\Desktop\plant class\withbackgroundclass"

train_dataset = datasets.ImageFolder(data_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(data_dir, transform=val_transform)

if train_dataset.classes != val_dataset.classes:
    raise ValueError("Training and validation datasets must have the same class folders.")

dataset_size = len(train_dataset)
if dataset_size < 2:
    raise ValueError("At least two images are required for a train/validation split.")

train_size = max(1, int(0.8 * dataset_size))
val_size = dataset_size - train_size
split = torch.utils.data.random_split(
    range(dataset_size),
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_data = torch.utils.data.Subset(train_dataset, split[0].indices)
val_data = torch.utils.data.Subset(val_dataset, split[1].indices)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=32, shuffle=False)

num_classes = len(train_dataset.classes)
print(f"Images: {dataset_size} | Training: {train_size} | Validation: {val_size}")

Images: 2420 | Training: 1936 | Validation: 484


In [5]:
model = models.efficientnet_b0(pretrained=True)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(device)

c:\Users\harsh\anaconda3\envs\gpu_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\harsh\anaconda3\envs\gpu_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [7]:
for epoch in range(10):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()

    train_loss = running_loss / len(train_data)
    train_accuracy = correct / len(train_data)

    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for val_images, val_labels in val_loader:
            val_images, val_labels = val_images.to(device), val_labels.to(device)
            val_outputs = model(val_images)
            val_loss = criterion(val_outputs, val_labels)

            val_running_loss += val_loss.item() * val_labels.size(0)
            val_correct += (val_outputs.argmax(dim=1) == val_labels).sum().item()

    val_loss = val_running_loss / len(val_data)
    val_accuracy = val_correct / len(val_data)

    print(
        f"Epoch {epoch + 1}/10 | "
        f"Loss: {train_loss:.4f} | Accuracy: {train_accuracy:.2%} | "
        f"Val Loss: {val_loss:.4f} | Val Accuracy: {val_accuracy:.2%}"
    )

Epoch 1/10 | Loss: 1.2831 | Accuracy: 67.56% | Val Loss: 0.4377 | Val Accuracy: 87.81%
Epoch 2/10 | Loss: 0.3039 | Accuracy: 91.37% | Val Loss: 0.2681 | Val Accuracy: 92.15%
Epoch 3/10 | Loss: 0.2028 | Accuracy: 94.11% | Val Loss: 0.1920 | Val Accuracy: 93.60%
Epoch 4/10 | Loss: 0.1640 | Accuracy: 94.99% | Val Loss: 0.1238 | Val Accuracy: 94.83%
Epoch 5/10 | Loss: 0.1270 | Accuracy: 96.07% | Val Loss: 0.1161 | Val Accuracy: 96.28%
Epoch 6/10 | Loss: 0.1423 | Accuracy: 95.51% | Val Loss: 0.1856 | Val Accuracy: 94.83%
Epoch 7/10 | Loss: 0.1163 | Accuracy: 96.33% | Val Loss: 0.0696 | Val Accuracy: 97.31%
Epoch 8/10 | Loss: 0.1022 | Accuracy: 97.42% | Val Loss: 0.1519 | Val Accuracy: 95.04%
Epoch 9/10 | Loss: 0.0781 | Accuracy: 97.78% | Val Loss: 0.1772 | Val Accuracy: 95.25%
Epoch 10/10 | Loss: 0.0790 | Accuracy: 97.73% | Val Loss: 0.1160 | Val Accuracy: 96.07%


In [8]:
print("Model:", next(model.parameters()).device)
print("Images:", images.device)
print("Outputs:", outputs.device)

Model: cuda:0
Images: cuda:0
Outputs: cuda:0


In [9]:
for param in model.features.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=1e-5)

for epoch in range(10):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()

    train_loss = running_loss / len(train_data)
    train_accuracy = correct / len(train_data)

    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for val_images, val_labels in val_loader:
            val_images, val_labels = val_images.to(device), val_labels.to(device)
            val_outputs = model(val_images)
            val_loss = criterion(val_outputs, val_labels)

            val_running_loss += val_loss.item() * val_labels.size(0)
            val_correct += (val_outputs.argmax(dim=1) == val_labels).sum().item()

    val_loss = val_running_loss / len(val_data)
    val_accuracy = val_correct / len(val_data)

    print(
        f"Fine-tune Epoch {epoch + 1}/10 | "
        f"Loss: {train_loss:.4f} | Accuracy: {train_accuracy:.2%} | "
        f"Val Loss: {val_loss:.4f} | Val Accuracy: {val_accuracy:.2%}"
    )

Fine-tune Epoch 1/10 | Loss: 0.0520 | Accuracy: 98.76% | Val Loss: 0.0637 | Val Accuracy: 98.14%
Fine-tune Epoch 2/10 | Loss: 0.0320 | Accuracy: 99.28% | Val Loss: 0.0597 | Val Accuracy: 98.35%
Fine-tune Epoch 3/10 | Loss: 0.0380 | Accuracy: 99.02% | Val Loss: 0.0492 | Val Accuracy: 98.35%
Fine-tune Epoch 4/10 | Loss: 0.0229 | Accuracy: 99.43% | Val Loss: 0.0485 | Val Accuracy: 98.55%
Fine-tune Epoch 5/10 | Loss: 0.0219 | Accuracy: 99.33% | Val Loss: 0.0491 | Val Accuracy: 98.35%
Fine-tune Epoch 6/10 | Loss: 0.0288 | Accuracy: 99.38% | Val Loss: 0.0446 | Val Accuracy: 98.97%
Fine-tune Epoch 7/10 | Loss: 0.0325 | Accuracy: 99.23% | Val Loss: 0.0451 | Val Accuracy: 98.14%
Fine-tune Epoch 8/10 | Loss: 0.0210 | Accuracy: 99.54% | Val Loss: 0.0451 | Val Accuracy: 98.14%
Fine-tune Epoch 9/10 | Loss: 0.0129 | Accuracy: 99.74% | Val Loss: 0.0402 | Val Accuracy: 98.55%
Fine-tune Epoch 10/10 | Loss: 0.0137 | Accuracy: 99.74% | Val Loss: 0.0330 | Val Accuracy: 99.17%


In [10]:
torch.save(model.state_dict(), "efficientnet_plants.wbcth")